# BuzzGuard - Phase 0: train the wingbeat classifier

**Goal of this notebook:** find out, fast, whether a small CNN can tell mosquito species apart
from wingbeat audio - and produce a real confusion matrix you can defend in front of a jury.

This is the kill-switch step. If the model does not work, you want to know now, not in week 6.

**What you get at the end:**
1. A trained model (Aedes / other mosquito / background noise)
2. A confusion matrix PNG, on a held-out test set, for your poster
3. A TensorFlow.js export ready to drop into the Phase 1 web app

---

### Read this before you run anything

- **The training data is lab-recorded and clean.** Wingbeats was captured in controlled conditions.
  Your phone, in a room, with a fan on, is not that. Expect real-world accuracy to be **much lower**
  than what you see here. That gap is not a bug in your work - it is the honest finding, and saying so
  out loud is what separates you from a re-implementation.
- **Wingbeats has no "no mosquito" class.** It is mosquito-only. We add a background class from
  ESC-50 (environmental sounds) so the model can learn to say "nothing here". Without this, your app
  will confidently classify silence as a mosquito.
- **Numbers from this notebook are lab numbers.** Do not put them on your poster as if they were
  field results. Label them as such, then go get field numbers in week 5.
- This notebook was written but **not executed end-to-end** by the person who drafted it
  (no Kaggle access from that environment). Run the smoke test in section 2 first - it is designed
  to catch export problems before you spend an hour training.

## 1. Setup

Runtime > Change runtime type > **T4 GPU** before you start.

**About these version pins - the order genuinely matters.**

If you ask pip for `tensorflowjs` and `protobuf==6.31.1` in one command, it cannot satisfy both with a
modern tensorflowjs, so it silently **backtracks to tensorflowjs 3.18.0** (from 2022). That version
calls `np.object`, which NumPy removed in 1.24, and the import dies with
`AttributeError: module 'numpy' has no attribute 'object'`. The pin is not the problem - the
resolver quietly downgrading you is.

So: install tensorflowjs **first, pinned**, then fix protobuf/setuptools **after**. Tested working:
**tensorflowjs 4.22 + protobuf 6.31.1 + setuptools 80.x + NumPy 2.1** (tensorflowjs may pull TF down
to 2.19; that is fine and expected).

Run this cell, then **Runtime > Restart session**, then continue from the next cell.

In [ ]:
!pip -q install librosa soundfile
!pip -q install tensorflowjs==4.22.0
!pip -q install "protobuf==6.31.1" "setuptools<81"
# ^ then: Runtime > Restart session, and continue from the next cell.

import os, glob, json, random, pathlib, collections
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 1337
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

import google.protobuf as _pb
print("TF", tf.__version__, "| protobuf", _pb.__version__, "| numpy", np.__version__)
try:
    import tensorflowjs as tfjs
    print("tensorflowjs", tfjs.__version__, "- import OK")
    if int(tfjs.__version__.split(".")[0]) < 4:
        print("   ^^ TOO OLD. pip backtracked. Rerun the install cell exactly as written")
        print("      (tensorflowjs FIRST and pinned, protobuf AFTER), then restart.")
except AttributeError as e:
    print("tensorflowjs import FAILED (np.object) ->", str(e)[:90])
    print("   You have an ancient tensorflowjs (3.x). pip backtracked because protobuf")
    print("   was pinned in the same command. Rerun the install cell, then restart.")
except Exception as e:
    print("tensorflowjs import FAILED ->", type(e).__name__, str(e)[:150])
    print("   fix: rerun the install cell, then Runtime > Restart session")
print("GPU:", tf.config.list_physical_devices("GPU") or "NONE - go enable the T4")

In [ ]:
# ----- knobs -----
SR         = 8000    # Wingbeats is 8 kHz; phones can record this easily
CLIP       = 5000    # samples per clip = 0.625 s
N_FFT      = 256
HOP        = 64
N_MELS     = 64
FMIN, FMAX = 50.0, 4000.0

MAX_PER_CLASS = 6000   # cap per class for a fast first run. Raise to 20000+ once it works.
BATCH         = 64
EPOCHS        = 25

# Split by recording folder instead of by file.
# True  = harder, honest number (no session leakage)
# False = random split, inflated number
GROUP_SPLIT_BY_FOLDER = True

# 3-class default, matching the build plan.
# For the 4-class matrix on your poster, split "other_mosquito" into anopheles / culex.
SPECIES_TO_CLASS = {
    "Ae. aegypti":          "aedes",
    "Ae. albopictus":       "aedes",
    "An. arabiensis":       "other_mosquito",
    "An. gambiae":          "other_mosquito",
    "C. pipiens":           "other_mosquito",
    "C. quinquefasciatus":  "other_mosquito",
}
CLASSES = ["aedes", "other_mosquito", "background"]
NUM_CLASSES = len(CLASSES)

FRAMES = (CLIP - N_FFT) // HOP + 1
print("mel-spectrogram shape per clip:", (FRAMES, N_MELS))

## 2. Model + export smoke test (run this BEFORE training)

The mel-spectrogram is computed **inside the model** using `tf.signal` ops.

Why this matters: the classic way to fail is to compute mel-spectrograms with librosa in Python,
then compute them slightly differently in JavaScript on the phone. The model then sees inputs it was
never trained on and accuracy quietly collapses. Putting the transform in the graph means the phone
runs the *same* ops. The model takes raw audio in, gives class probabilities out.

**Is that safe?** `tf.signal.stft` does not always survive conversion to TensorFlow.js, so this was
tested before being recommended: with the pinned versions above, the model converts cleanly to a
`tfjs_graph_model`, the FFT lands in the graph as an `RFFT` op (which TF.js supports), and the whole
bundle is about **0.3 MB**. It works.

We still smoke-test it here with an *untrained* model, because your Colab versions may differ from the
tested ones. If it fails, you have lost 30 seconds instead of an hour of training.

In [ ]:
class MelSpectrogram(layers.Layer):
    """Raw waveform -> log-mel spectrogram, as graph ops."""
    def __init__(self, sr=SR, n_fft=N_FFT, hop=HOP, n_mels=N_MELS,
                 fmin=FMIN, fmax=FMAX, **kw):
        super().__init__(**kw)
        self.sr, self.n_fft, self.hop = sr, n_fft, hop
        self.n_mels, self.fmin, self.fmax = n_mels, fmin, fmax

    def build(self, input_shape):
        self.mel_w = tf.signal.linear_to_mel_weight_matrix(
            num_mel_bins=self.n_mels,
            num_spectrogram_bins=self.n_fft // 2 + 1,
            sample_rate=self.sr,
            lower_edge_hertz=self.fmin,
            upper_edge_hertz=self.fmax)
        super().build(input_shape)

    def call(self, x):
        stft = tf.signal.stft(x, frame_length=self.n_fft,
                              frame_step=self.hop, fft_length=self.n_fft)
        spec = tf.abs(stft)
        mel = tf.tensordot(spec, self.mel_w, 1)
        log_mel = tf.math.log(mel + 1e-6)
        # per-example standardisation: makes it robust to mic gain / loudness
        mean = tf.reduce_mean(log_mel, axis=[1, 2], keepdims=True)
        std = tf.math.reduce_std(log_mel, axis=[1, 2], keepdims=True) + 1e-5
        return (log_mel - mean) / std

    def get_config(self):
        c = super().get_config()
        c.update(dict(sr=self.sr, n_fft=self.n_fft, hop=self.hop,
                      n_mels=self.n_mels, fmin=self.fmin, fmax=self.fmax))
        return c


def build_model(num_classes=NUM_CLASSES):
    inp = keras.Input(shape=(CLIP,), name="waveform")
    x = MelSpectrogram()(inp)
    x = layers.Reshape((FRAMES, N_MELS, 1))(x)
    for f in (16, 32, 64, 64):
        x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D(2)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation="softmax", name="species")(x)
    return keras.Model(inp, out, name="buzzguard")

m = build_model()
m.summary()
print("\nparams:", m.count_params())

In [ ]:
# ---- SMOKE TEST: can this model reach the phone at all? ----
import subprocess, shutil
os.makedirs("/content/smoke", exist_ok=True)
m.export("/content/smoke/saved") if hasattr(m, "export") else m.save("/content/smoke/saved")

r = subprocess.run(
    ["tensorflowjs_converter", "--input_format=tf_saved_model",
     "--output_format=tfjs_graph_model",
     "/content/smoke/saved", "/content/smoke/tfjs"],
    capture_output=True, text=True)

print("STDOUT:", r.stdout[-2000:])
print("STDERR:", r.stderr[-2000:])
if r.returncode == 0:
    print("\n=> EXPORT PATH OK. Safe to train.")
else:
    print("\n=> EXPORT FAILED. Read the fallback note in the next cell BEFORE training.")

### Numbers for your poster

This architecture is about **61,000 parameters** and exports to roughly **0.3 MB** of TF.js.

Your poster currently claims "~120k params" and "<2 MB" - both were placeholders. The real numbers are
better than the guesses, so update them. Anything on that board you cannot reproduce on demand is a
liability in front of a jury.

### If the smoke test failed

Do not panic and do not start training. Most likely cause is a version mismatch - rerun the install
cell and restart the session first. If it still fails, you have two options:

**Plan B - move the mel transform out of the model.** Delete the `MelSpectrogram()` line from
`build_model` and make the input `shape=(FRAMES, N_MELS, 1)`. Compute mel-spectrograms with librosa
during training, and replicate them in JavaScript on the phone using
[Meyda](https://meyda.js.org/) or a hand-rolled FFT.
Cost: you must match librosa's mel filterbank in JS *exactly*, or accuracy drops for reasons that are
maddening to debug. If you go this route, verify by feeding one identical WAV through both paths and
comparing the arrays before you trust anything.

**Plan C - keep the transform in the model, but do the STFT in JS.** TensorFlow.js has
`tf.signal.stft`. Export a model that takes the spectrogram, and do STFT + mel in JS with tfjs ops.

**If it still fails, do not let it block you.** The export only matters for getting the model onto a
phone in Phase 1. Training does not need it. Comment out the smoke test, run sections 3 to 6, and get
your confusion matrix - that is the actual Phase 0 deliverable. The `.keras` file saves regardless,
and you can convert it later in a clean notebook.

## 3. Get the data

Kaggle has moved away from the old `kaggle.json` file. Tokens now look like `KGAT_xxxxxxxx` and are
supplied either as the `KAGGLE_API_TOKEN` environment variable or in `~/.kaggle/access_token`.

Get one at: kaggle.com > Settings > API > **Generate New Token**. Copy it immediately - Kaggle shows
it once and never again.

Two gotchas:
- **Colab ships an old kaggle client** that only understands `kaggle.json`, so the cell below upgrades
  it first. Without the upgrade your token will be rejected no matter what you do.
- The prompt below is a **password box** - your token will not be echoed or saved into the notebook.
  Do not paste the token into a normal cell, or it gets stored in the file and travels with it.

Wingbeats is a few GB. The download takes a while.

In [ ]:
!pip -q install -U kaggle

import os, getpass
tok = getpass.getpass("Paste your Kaggle token (starts with KGAT_): ").strip()

# both mechanisms, belt and braces
os.environ["KAGGLE_API_TOKEN"] = tok
kdir = os.path.expanduser("~/.kaggle")
os.makedirs(kdir, exist_ok=True)
with open(os.path.join(kdir, "access_token"), "w") as f:
    f.write(tok)
os.chmod(os.path.join(kdir, "access_token"), 0o600)

import subprocess
print("kaggle client:", subprocess.run(["kaggle","--version"],
      capture_output=True, text=True).stdout.strip())
del tok

In [ ]:
!kaggle datasets download -d potamitis/wingbeats -p /content/data --unzip
print("done")

In [ ]:
# Inspect what actually landed on disk. Do not trust an assumed folder layout.
root = "/content/data"
for dirpath, dirnames, filenames in os.walk(root):
    depth = dirpath.replace(root, "").count(os.sep)
    if depth <= 2:
        wavs = [f for f in filenames if f.lower().endswith(".wav")]
        print("  " * depth, os.path.basename(dirpath) or ".",
              f"[{len(dirnames)} dirs, {len(wavs)} wavs]")
    if depth > 2:
        dirnames[:] = []

In [ ]:
# Build the file index. Globs recursively, so layout quirks do not break it.
def find_species_dir(root, species):
    hits = [p for p in glob.glob(os.path.join(root, "**", species), recursive=True)
            if os.path.isdir(p)]
    return hits[0] if hits else None

index = []   # (filepath, class_name, group_id)
missing = []
for species, cls in SPECIES_TO_CLASS.items():
    d = find_species_dir(root, species)
    if d is None:
        missing.append(species); continue
    wavs = glob.glob(os.path.join(d, "**", "*.wav"), recursive=True)
    for w in wavs:
        # group = the immediate parent folder (a recording session)
        group = os.path.basename(os.path.dirname(w))
        index.append((w, cls, f"{species}/{group}"))
    print(f"{species:24s} -> {cls:16s} {len(wavs):7d} files")

if missing:
    print("\nNOT FOUND (check the tree printed above and fix SPECIES_TO_CLASS):", missing)
print("\ntotal mosquito clips:", len(index))

### The background class

Wingbeats is mosquito-only. We pull environmental sounds from ESC-50 for the "nothing here" class.

Note the trap: ESC-50 contains an `insects` category. Leaving it in teaches the model that
insect-like buzzing is background - the exact opposite of what you want. We drop it by default.

In [ ]:
!git clone -q --depth 1 https://github.com/karolpiczak/ESC-50.git /content/esc50

import csv, librosa

meta = list(csv.DictReader(open("/content/esc50/meta/esc50.csv")))
DROP_CATEGORIES = {"insects"}   # would poison the background class

bg_files = [m["filename"] for m in meta if m["category"] not in DROP_CATEGORIES]
print("ESC-50 clips used:", len(bg_files), "| dropped categories:", DROP_CATEGORIES)

def load_background(n_needed):
    """ESC-50 is 5 s @ 44.1 kHz. Resample to 8 kHz, chop into CLIP-sized pieces."""
    out = []
    random.shuffle(bg_files)
    for fn in bg_files:
        y, _ = librosa.load(f"/content/esc50/audio/{fn}", sr=SR, mono=True)
        for i in range(0, len(y) - CLIP, CLIP):
            out.append(y[i:i + CLIP].astype(np.float32))
            if len(out) >= n_needed:
                return out
    return out

In [ ]:
# Cap per class, then load audio into memory.
by_class = collections.defaultdict(list)
for f, c, g in index:
    by_class[c].append((f, g))

for c in by_class:
    random.shuffle(by_class[c])
    by_class[c] = by_class[c][:MAX_PER_CLASS]
    print(f"{c:16s} {len(by_class[c])}")

def load_clip(path):
    y, _ = librosa.load(path, sr=SR, mono=True)
    if len(y) < CLIP:
        y = np.pad(y, (0, CLIP - len(y)))
    return y[:CLIP].astype(np.float32)

X, Y, G = [], [], []
for c in ["aedes", "other_mosquito"]:
    for f, g in by_class[c]:
        X.append(load_clip(f)); Y.append(CLASSES.index(c)); G.append(g)
    print("loaded", c, len(X))

n_bg = min(MAX_PER_CLASS, max(1, len(X) // 2))
for i, clip in enumerate(load_background(n_bg)):
    X.append(clip); Y.append(CLASSES.index("background")); G.append(f"esc50/{i % 50}")

X = np.stack(X); Y = np.array(Y); G = np.array(G)
print("\nX", X.shape, "| class counts:", collections.Counter([CLASSES[i] for i in Y]))

## 4. Split

`GROUP_SPLIT_BY_FOLDER = True` keeps every clip from one recording session entirely inside one split.

This matters more than it looks. With a random split, clips recorded seconds apart - same mosquito,
same mic, same room tone - land in both train and test. The model learns the *session*, not the
species, and hands you a beautiful, meaningless 99%. Splitting by group gives a lower number that is
actually about mosquitoes.

If a judge asks "how do you know it generalises?", this is your answer.

In [ ]:
from sklearn.model_selection import train_test_split

if GROUP_SPLIT_BY_FOLDER:
    groups = np.unique(G)
    g_tr, g_tmp = train_test_split(groups, test_size=0.3, random_state=SEED)
    g_va, g_te = train_test_split(g_tmp, test_size=0.5, random_state=SEED)
    tr = np.isin(G, g_tr); va = np.isin(G, g_va); te = np.isin(G, g_te)
    idx_tr, idx_va, idx_te = np.where(tr)[0], np.where(va)[0], np.where(te)[0]
    print("split by recording group (honest)")
else:
    i_all = np.arange(len(X))
    idx_tr, i_tmp = train_test_split(i_all, test_size=0.3, random_state=SEED, stratify=Y)
    idx_va, idx_te = train_test_split(i_tmp, test_size=0.5, random_state=SEED, stratify=Y[i_tmp])
    print("random split (optimistic - expect inflated accuracy)")

for name, idx in [("train", idx_tr), ("val", idx_va), ("test", idx_te)]:
    print(f"{name:6s} {len(idx):6d}  {collections.Counter([CLASSES[i] for i in Y[idx]])}")

In [ ]:
def augment(x, y):
    """Noise, gain and time-shift. This is what buys you real-world robustness."""
    x = tf.roll(x, shift=tf.random.uniform([], -400, 400, dtype=tf.int32), axis=0)
    x = x * tf.random.uniform([], 0.5, 1.5)                      # mic gain
    x = x + tf.random.normal(tf.shape(x), stddev=tf.random.uniform([], 0.0, 0.05))
    return x, y

def make_ds(idx, training):
    ds = tf.data.Dataset.from_tensor_slices((X[idx], Y[idx]))
    if training:
        ds = ds.shuffle(4096, seed=SEED).map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

ds_tr, ds_va, ds_te = make_ds(idx_tr, True), make_ds(idx_va, False), make_ds(idx_te, False)

cnt = collections.Counter(Y[idx_tr])
total = sum(cnt.values())
class_weight = {i: total / (NUM_CLASSES * cnt[i]) for i in range(NUM_CLASSES) if cnt[i]}
print("class weights:", {CLASSES[k]: round(v, 2) for k, v in class_weight.items()})

## 5. Train

In [ ]:
model = build_model()
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

cbs = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=6,
                                  restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                      patience=3, verbose=1),
]

hist = model.fit(ds_tr, validation_data=ds_va, epochs=EPOCHS,
                 class_weight=class_weight, callbacks=cbs)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist.history["accuracy"], label="train")
ax[0].plot(hist.history["val_accuracy"], label="val")
ax[0].set_title("accuracy"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(hist.history["loss"], label="train")
ax[1].plot(hist.history["val_loss"], label="val")
ax[1].set_title("loss"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.savefig("/content/training_curves.png", dpi=150)
plt.show()

# A big train/val gap means memorisation. More augmentation or fewer params.

## 6. Evaluate - this is the deliverable

The confusion matrix is the artefact you want. Read it by row: of everything that really was
*Aedes*, where did the model put it?

Pay attention to the **Aedes recall** number. Missing a dengue vector is the expensive error.
Calling a harmless *Culex* an *Aedes* wastes a fogging trip; missing a real *Aedes* means the
outbreak proceeds. Optimise for recall on Aedes and say so explicitly in your write-up - that
reasoning is exactly what a jury wants to hear.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_true = Y[idx_te]
y_prob = model.predict(ds_te, verbose=0)
y_pred = y_prob.argmax(1)

print("TEST accuracy:", round((y_pred == y_true).mean(), 4))
print("\n", classification_report(y_true, y_pred, target_names=CLASSES, digits=3))

cm = confusion_matrix(y_true, y_pred)
cm_pct = cm / cm.sum(1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(5.5, 4.8))
im = ax.imshow(cm_pct, cmap="Greens", vmin=0, vmax=100)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASSES, rotation=20, ha="right")
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASSES)
ax.set_xlabel("predicted"); ax.set_ylabel("actual")
ax.set_title("BuzzGuard - confusion matrix (held-out test, %)")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, f"{cm_pct[i,j]:.0f}", ha="center", va="center",
                color="white" if cm_pct[i, j] > 55 else "#222", fontsize=11)
plt.tight_layout(); plt.savefig("/content/confusion_matrix.png", dpi=200)
plt.show()
print("saved -> /content/confusion_matrix.png  (this replaces the placeholder on your poster)")

In [ ]:
# Confidence threshold sweep: what does the gate on your app actually cost you?
print(f"{'thresh':>7} {'coverage':>9} {'acc on kept':>12}")
for t in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    keep = y_prob.max(1) >= t
    if keep.sum() == 0:
        continue
    acc = (y_pred[keep] == y_true[keep]).mean()
    print(f"{t:>7.2f} {keep.mean()*100:>8.1f}% {acc*100:>11.1f}%")

print("\nPick the threshold for Phase 1 from this table, and quote the tradeoff out loud.")

## 7. Export for the phone

In [ ]:
os.makedirs("/content/export", exist_ok=True)
model.save("/content/export/buzzguard.keras")

sm = "/content/export/saved"
model.export(sm) if hasattr(model, "export") else model.save(sm)

r = subprocess.run(
    ["tensorflowjs_converter", "--input_format=tf_saved_model",
     "--output_format=tfjs_graph_model", sm, "/content/export/tfjs"],
    capture_output=True, text=True)
print(r.stdout[-1500:]); print(r.stderr[-1500:])

if r.returncode == 0:
    total = sum(os.path.getsize(os.path.join(dp, f))
                for dp, _, fs in os.walk("/content/export/tfjs") for f in fs)
    print(f"\nTFJS export OK - {total/1e6:.2f} MB")
    print("files:", os.listdir("/content/export/tfjs"))

json.dump({"classes": CLASSES, "sr": SR, "clip": CLIP,
           "n_fft": N_FFT, "hop": HOP, "n_mels": N_MELS,
           "fmin": FMIN, "fmax": FMAX},
          open("/content/export/model_meta.json", "w"), indent=2)

shutil.make_archive("/content/buzzguard_phase0", "zip", "/content/export")

from google.colab import files
files.download("/content/buzzguard_phase0.zip")
files.download("/content/confusion_matrix.png")

## 8. What you now have, and what you do not

**Have:**
- A model that separates Aedes from other mosquitoes and from background, on lab audio
- An honest confusion matrix from a session-grouped split
- A TF.js bundle for Phase 1

**Do not have - and must not claim:**
- Any evidence this works on a real phone, in a real room, on real Indian mosquitoes.
  Lab-recorded audio is the easy case.

**Next moves, in order:**
1. Note the test accuracy and Aedes recall. Put them on your poster **labelled as lab results**.
2. Phase 1: wire this into the web app, mic to prediction.
3. Week 5 is the one that wins it: record actual mosquitoes near you, run this model on them,
   and report what happened - including the drop. "We got 71% in the field, here is why"
   beats "papers report 97%" every single time, because the first one is yours.

**If accuracy is bad (<70%):** try MAX_PER_CLASS higher, EPOCHS higher, or check that
`find_species_dir` actually located every species (a silent miss shows up as a missing class).
If aedes-vs-other is near chance but background is perfect, the species signal is not surviving
your preprocessing - inspect a few mel-spectrograms by eye before touching the model.